# 05 - Decision numbers

The data has **no promotion cost and no margin**, so profit cannot be measured. What it does give is the *extra sales* the model attributes to a promo day.
This notebook turns that into two things a decision-maker can use:

1. **Break-even cost:** the most a promotion can cost per store per promo day before it stops paying for itself, at a few assumed gross margins.
   Compare that number with the real cost and the keep / change / stop answer follows.
2. **An indicative chain-wide figure,** scaled from the 50-store sample to all 1,115 stores. This is an order of magnitude, not a forecast (details in the caveats below).

Everything here is built from `uplift_by_segment.csv` (Stage 3) and the cleaned data. Nothing new is estimated.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW, CLEAN, ANALYTICS = Path("../data/raw"), Path("../data/clean"), Path("../data/analytics")
seg = pd.read_csv(ANALYTICS / "uplift_by_segment.csv")
seg = seg[seg.segment_type == "store_type"].set_index("segment")
store_all = pd.read_csv(RAW / "store.csv")
sales = pd.read_csv(CLEAN / "sales_clean.csv", parse_dates=["Date"])

years = ((sales.Date.max() - sales.Date.min()).days + 1) / 365.25
promo_days_per_store_year = sales[sales.Promo == 1].groupby("Store").size().mean() / years
print(f"data span {years:.2f} years | promo days per store per year (open days): {promo_days_per_store_year:.0f}")

data span 2.58 years | promo days per store per year (open days): 130


## Impact by store type

- `incremental_share`: of the sales made on a promo day, the share that is extra (uplift / (1 + uplift)). A 35% uplift means about a quarter of promo-day sales would not have happened.
- `breakeven_*`: extra sales per promo day x assumed gross margin. **The margins (10%, 20%, 30%) are illustrative assumptions, not data.** Replace them with the client's real margin.
- `indicative_annual_extra_revenue`: chain stores of that type x promo days per store-year x extra sales per promo day.

In [2]:
MARGINS = [0.10, 0.20, 0.30]
rows = []
for t in ["a", "b", "c", "d"]:
    up = seg.loc[t, "model_uplift_pct"]
    extra = seg.loc[t, "avg_extra_sales_per_promo_day"]
    chain_n = int((store_all.StoreType == t).sum())
    rows.append({
        "StoreType": t, "sample_stores": int(seg.loc[t, "stores"]), "chain_stores": chain_n,
        "model_uplift_pct": up, "extra_sales_per_promo_day": extra,
        "incremental_share_pct": up / (100 + up) * 100,
        **{f"breakeven_cost_{int(m * 100)}pct_margin": extra * m for m in MARGINS},
        "indicative_annual_extra_revenue": chain_n * promo_days_per_store_year * extra,
    })
impact = pd.DataFrame(rows)
total_row = {"StoreType": "all", "sample_stores": impact.sample_stores.sum(), "chain_stores": impact.chain_stores.sum(),
             "indicative_annual_extra_revenue": impact.indicative_annual_extra_revenue.sum()}
impact_all = pd.concat([impact, pd.DataFrame([total_row])], ignore_index=True)
impact_all.round(1)

,StoreType,sample_stores,chain_stores,model_uplift_pct,extra_sales_per_promo_day,incremental_share_pct,breakeven_cost_10pct_margin,breakeven_cost_20pct_margin,breakeven_cost_30pct_margin,indicative_annual_extra_revenue
0,a,24,602,37.2,2414.9,27.1,241.5,483.0,724.5,189580233.6
1,b,4,17,19.6,1432.2,16.4,143.2,286.4,429.7,3175006.4
2,c,8,148,38.0,2063.4,27.5,206.3,412.7,619.0,39823485.0
3,d,14,348,35.3,2147.9,26.1,214.8,429.6,644.4,97473221.1
4,all,50,1115,NaN,NaN,NaN,NaN,NaN,NaN,330051946.1


## How much worse is type b?

Type b earns less extra revenue per promo day than the other types. If a promotion costs the same to run in every store, type b pays back the least.

In [3]:
others = impact[impact.StoreType != "b"]
w_other = np.average(others.extra_sales_per_promo_day, weights=others.chain_stores)
b = impact.set_index("StoreType").loc["b"]
print(f"type b extra sales per promo day: {b.extra_sales_per_promo_day:,.0f} EUR")
print(f"other types (chain-store-weighted): {w_other:,.0f} EUR")
print(f"type b earns {(1 - b.extra_sales_per_promo_day / w_other) * 100:.0f}% less extra revenue per promo day than the other types")
print(f"share of chain stores that are type b: {b.chain_stores / impact.chain_stores.sum() * 100:.1f}%")
print(f"share of the indicative annual extra revenue from type b: "
      f"{b.indicative_annual_extra_revenue / impact.indicative_annual_extra_revenue.sum() * 100:.1f}%")

type b extra sales per promo day: 1,432 EUR
other types (chain-store-weighted): 2,283 EUR
type b earns 37% less extra revenue per promo day than the other types
share of chain stores that are type b: 1.5%
share of the indicative annual extra revenue from type b: 1.0%


## Save

In [4]:
impact_all.round(2).to_csv(ANALYTICS / "decision_impact.csv", index=False)
print("saved", ANALYTICS / "decision_impact.csv")

saved ../data/analytics/decision_impact.csv


## Caveats for the indicative chain-wide figure
- It scales one 6-week window (late June to July) to a full year. Promotions may work differently in other seasons.
- It assumes the 50-store sample represents each store type across the chain, and that a store runs about 130 promo days a year, as in this data.
- It is **extra revenue, not profit**: promotion cost and margin are not available.
- The uplift behind it is a model-based estimate from observational data, not a randomised experiment.
Treat it as "order of magnitude, low confidence". The break-even column is more robust because it makes no claim about what a promotion costs.